# Blood Work Analysis Pipeline
A two-stage LLM pipeline:
- **Stage 1:** Extract and flag abnormal values from a blood report
- **Stage 2:** Generate a health summary and Indian diet plan based on flagged values

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

## Sample Blood Report

In [4]:
with open("blood_work.txt", "r") as f:
    blood_report = f.read()

print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5–17.5)
Hematocrit:        44%            


## Stage 1 — Extract and Flag Abnormal Values

In [5]:
extraction_prompt = f"""
You are a medical data extraction assistant.

From the blood report below, extract ALL test values and classify each one as HIGH, LOW, or NORMAL 
based on the reference ranges provided in the report.

Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text

print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

=== STAGE 1: EXTRACTED VALUES ===
Here are the extracted test values and their classifications:

- Hemoglobin: 15.1 g/dL | Status: NORMAL | Reference: 13.5–17.5
- Hematocrit: 44% | Status: NORMAL | Reference: 41–53%
- WBC: 6.8 x10^3/uL | Status: NORMAL | Reference: 4.5–11.0
- Platelets: 220 x10^3/uL | Status: NORMAL | Reference: 150–400
- Total Cholesterol: 238 mg/dL | Status: HIGH | Reference: <200
- LDL Cholesterol: 162 mg/dL | Status: HIGH | Reference: <100
- HDL Cholesterol: 36 mg/dL | Status: LOW | Reference: >40
- Triglycerides: 188 mg/dL | Status: HIGH | Reference: <150
- Glucose (Fasting): 92 mg/dL | Status: NORMAL | Reference: 70–99
- HbA1c: 5.3% | Status: NORMAL | Reference: <5.7%
- Creatinine: 1.0 mg/dL | Status: NORMAL | Reference: 0.7–1.3
- eGFR: 82 mL/min | Status: NORMAL | Reference: >60
- ALT: 28 U/L | Status: NORMAL | Reference: 7–40
- AST: 25 U/L | Status: NORMAL | Reference: 10–40
- Bilirubin Total: 0.8 mg/dL | Status: NORMAL | Reference: 0.2–1.2


## Stage 2 — Health Summary and Indian Diet Plan

In [6]:
diet_prompt = f"""
You are a clinical nutritionist specializing in Indian dietary habits.

Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines explaining the patient's condition in simple language
2. A short, practical Indian diet plan having only two sections (1) Foods to avoid (2) Foods to eat more of. 
   Do not include any other sections in diet plan.

Blood Work Analysis:
{extracted_values}
"""

diet_response = llm.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.text)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
Here is your health summary and diet plan:

### 1. Health Summary

Your blood tests show good overall health with normal blood sugar, kidney, and liver function. However, your cholesterol levels need attention. Your "bad" cholesterol (LDL) and triglycerides are high, while your "good" cholesterol (HDL) is low. This pattern increases your risk for heart disease, and targeted dietary changes are crucial to improve your heart health.

### 2. Practical Indian Diet Plan

**Foods to Avoid:**

*   **Deep-fried items:** Samosas, pakoras, puri, bhajiya, chips, store-bought namkeens.
*   **High-fat dairy & meats:** Full-cream milk, full-fat paneer, mawa, red meat (mutton, lamb), processed meats.
*   **Sugary foods & drinks:** Mithai (traditional sweets), sugary beverages, packaged juices, biscuits made with refined flour.
*   **Excessive ghee, butter, and coconut oil.**
*   **Refined flour products:** Naan, parotta, maida-based breads, cakes, pastries.